# Lecture 6 · Probabilistic Models and Learning II: Bayesian Machine Learning and Probabilistic Programming — Exercises

Companion to the note [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).

Practise the material of **Lecture 6**: plate notation, generative modelling and **Box's loop** (posterior predictive checks), **Bayesian linear and
logistic regression** (priors on weights, epistemic vs. aleatoric uncertainty), and approximate inference by **variational inference**: the KL divergence,
the **ELBO** with $\log p(\mathcal D) = \text{ELBO}(\phi) + D_{\text{KL}}(q_\phi\Vert p(W\mid\mathcal D))$, and mean-field Gaussian guides like Pyro's `AutoNormal`.
Pyro/PyTorch are not needed: every exercise is done by hand or in NumPy/SciPy, including a small stochastic-variational-inference loop with the
reparameterisation trick, i.e. what `SVI(model, guide, Adam, Trace_ELBO())` does under the hood.

**19 exercises** · 🧠 Concept ×5 · ✍️ By hand ×6 · 💻 Code ×8

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import matplotlib.pyplot as plt
from scipy.stats import norm, multivariate_normal, skew
from scipy.special import expit
from scipy.integrate import quad
from sklearn.linear_model import Ridge, LogisticRegression
rng = np.random.default_rng(6)

## Part A · Concepts

### Exercise 1 · Plates: what is inside, what is outside?
*🧠 Concept · ★☆☆*

Draw (or describe) the plate diagram of Bayesian linear regression $w\sim\mathcal N(\mu_0,\Sigma_0)$, $Y_i\mid w, x_i\sim\mathcal N(w^Tx_i,\sigma^2)$, $i = 1..N$.
(a) Which nodes are inside the plate, which outside? Which are observed (shaded)? (b) What conditional-independence statement does the plate encode?
(c) Which Pyro construct corresponds to the plate, and why is it faster than a Python `for` loop?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Global parameters are shared by all replicas; local variables exist once per data point.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Outside: $w$ (and the fixed hyper-parameters $\mu_0,\Sigma_0,\sigma^2$). Inside: $x_i$ (observed input) and $Y_i$ (observed, shaded); $w\to Y_i\leftarrow x_i$.
(b) The $Y_i$ are conditionally i.i.d. given $w$: $p(y_{1:N}\mid w, x_{1:N}) = \prod_ip(y_i\mid w, x_i)$.
(c) `with pyro.plate("data", N): pyro.sample("obs", dist.Normal(...), obs=y)`. It declares the conditional independence, so Pyro evaluates all $N$ log-densities as one
vectorised tensor operation (and can subsample mini-batches), instead of building $N$ separate sample sites (Exercise 13).

</details>

### Exercise 2 · Epistemic or aleatoric?
*🧠 Concept · ★☆☆*

Classify each source of uncertainty and say whether more data would reduce it:
(a) the spread of the posterior over the weights `w1` of a BNN; (b) the sensor noise in a thermometer; (c) the disagreement between sampled BNN regression lines
in a region with no training data; (d) the `data_variance = exp(log_sigma)` term in the BNN model.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Epistemic = about the model/parameters (reducible). Aleatoric = inherent in the data (irreducible).

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Epistemic, shrinks as $N\to\infty$. (b) Aleatoric, does not shrink. (c) Epistemic: lack of data in the gap (Exercise 16). (d) Aleatoric: it models the noise floor of the observations.
Exercise 15 shows the split numerically: the predictive variance is $\underbrace{\sigma^2}_{\text{aleatoric}} + \underbrace{x^T\Sigma_Nx}_{\text{epistemic}}$.

</details>

### Exercise 3 · Choosing an inference engine
*🧠 Concept · ★★☆*

Compare mean-field VI (`AutoNormal`), full-rank VI and MCMC on (i) accuracy of the posterior, (ii) cost, (iii) scaling to big data / GPUs.
Why does mean-field VI tend to **underestimate** posterior variance? Which direction of KL is minimised, and what does that imply?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

VI minimises $D_{\text{KL}}(q\Vert p) = E_q[\log q - \log p]$: it is very expensive for $q$ to put mass where $p$ has little.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Mean-field: fast, $O(D)$ variational parameters, mini-batch friendly; ignores correlations. Full-rank: captures correlations, $O(D^2)$ parameters and memory.
MCMC: asymptotically exact, no family restriction, but sequential and slow on large data.

$D_{\text{KL}}(q\Vert p)$ is "zero-forcing": wherever $p\approx0$, $q$ must be $\approx0$. A factorised $q$ cannot stretch along a correlated (diagonal) ridge of $p$, so it fits **inside**
it and becomes too narrow: its variances match the conditional, not the marginal, precision (Exercises 11 and 18).

</details>

### Exercise 4 · From Pyro code to mathematics
*🧠 Concept · ★★☆*

Write the generative model implemented by the BNN `model(x_data, y_data)` of the note as mathematics (priors, deterministic hidden layer, likelihood).
Then explain (a) why the scale is `torch.exp(log_sigma)` rather than `log_sigma`, (b) what `.to_event(2)` does to the log-probability, and
(c) the roles of `pyro.sample(..., obs=...)` and `pyro.param`.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Every `pyro.sample` without `obs` is a latent variable with a prior; the one with `obs` is the likelihood.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$W_1, b_1\in\mathbb{R}^{1\times250}$, $W_2\in\mathbb{R}^{250\times1}$, $b_2\in\mathbb{R}$ with all entries i.i.d. $\mathcal N(0,1)$; $\log\sigma\sim\mathcal N(0,1)$;
for each $i$ (plate): $h_i = \tanh(x_iW_1 + b_1)$, $\mu_i = h_iW_2 + b_2$, $y_i\sim\mathcal N(\mu_i, e^{\log\sigma})$.
(Note the code passes $e^{\log\sigma}$ as the Normal's **scale**, i.e. standard deviation, despite the name `data_variance`.)

(a) A scale must be positive; sampling/optimising $\log\sigma\in\mathbb{R}$ and exponentiating gives an unconstrained parameterisation, so no gradient step can produce an invalid value.
(b) It declares the rightmost 2 dimensions as one event: the log-probabilities of the 250 weights are **summed** into one joint log-density instead of being treated as 250 independent batch entries.
(c) `obs=` conditions the model on data (that site contributes the likelihood term); `pyro.param` registers a learnable (non-random) parameter, e.g. the guide's means and scales optimised by SVI.

</details>

### Exercise 5 · Criticising a model: posterior predictive checks
*🧠 Concept · ★★☆*

(a) Write the posterior predictive distribution used in the *Criticize* step of Box's loop. (b) Describe the recipe for a posterior predictive check with a test statistic $T$.
(c) A Gaussian model is fit to strongly right-skewed data. Which statistic would expose the misfit, and what would the PPC p-value look like?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Simulate replicated data sets $\mathcal X^{\text{rep}}$ from the fitted model and compare $T(\mathcal X^{\text{rep}})$ with $T(\mathcal X)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $p(X^{\text{new}}\mid\mathcal X) = \int p(X^{\text{new}}\mid\mathcal Z)p(\mathcal Z\mid\mathcal X)\,d\mathcal Z$.
(b) Repeat: draw $\mathcal Z\sim p(\mathcal Z\mid\mathcal X)$, draw $\mathcal X^{\text{rep}}\sim p(\cdot\mid\mathcal Z)$ of the same size, compute $T(\mathcal X^{\text{rep}})$.
The p-value is the fraction with $T(\mathcal X^{\text{rep}})\ge T(\mathcal X)$; values near 0 or 1 flag a systematic misfit.
(c) The skewness (or the maximum): Gaussian replicas are symmetric, so essentially none reach the observed skewness and the p-value is about 0, so revise the model (Exercise 19).

</details>

## Part B · By hand

### Exercise 6 · Log evidence = ELBO + KL
*✍️ By hand · ★★☆*

Prove $\log p(\mathcal D) = \text{ELBO}(\phi) + D_{\text{KL}}(q_\phi(W)\Vert p(W\mid\mathcal D))$ with
$\text{ELBO}(\phi) = E_{q}[\log p(\mathcal D, W)] - E_q[\log q_\phi(W)]$. Conclude (a) that the ELBO is a lower bound and (b) why maximising it over $\phi$ is equivalent to minimising the KL,
although the KL itself cannot be computed.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Write $\log p(W\mid\mathcal D) = \log p(\mathcal D, W) - \log p(\mathcal D)$ inside the KL.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$D_{\text{KL}} = E_q[\log q - \log p(W\mid\mathcal D)] = E_q[\log q - \log p(\mathcal D, W)] + \log p(\mathcal D) = -\text{ELBO} + \log p(\mathcal D)$, because $\log p(\mathcal D)$ does not depend on $W$.

(a) $D_{\text{KL}}\ge0$, so $\text{ELBO}\le\log p(\mathcal D)$, with equality iff $q = p(W\mid\mathcal D)$ (Exercise 17 checks this numerically).
(b) $\log p(\mathcal D)$ is a constant w.r.t. $\phi$, so $\arg\max_\phi\text{ELBO} = \arg\min_\phi D_{\text{KL}}$. The ELBO only needs the joint $p(\mathcal D, W)$, which we can evaluate; the intractable
evidence integral never appears.

</details>

### Exercise 7 · KL between two Gaussians
*✍️ By hand · ★☆☆*

For univariate Gaussians, $D_{\text{KL}}(\mathcal N(\mu_1,\sigma_1^2)\Vert\mathcal N(\mu_2,\sigma_2^2)) = \ln\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1-\mu_2)^2}{2\sigma_2^2} - \frac12$.
Compute $D_{\text{KL}}(\mathcal N(1, 1)\Vert\mathcal N(0, 4))$ and the reverse $D_{\text{KL}}(\mathcal N(0, 4)\Vert\mathcal N(1, 1))$. What does the asymmetry tell you?

In [ ]:
kl_forward = None   # KL(N(1,1) || N(0,4))
kl_reverse = None   # KL(N(0,4) || N(1,1))

_kl = lambda m1, s1, m2, s2: quad(lambda x: norm.pdf(x, m1, s1) * (norm.logpdf(x, m1, s1) - norm.logpdf(x, m2, s2)), -30, 30)[0]
check('KL(N(1,1)||N(0,4))', kl_forward, _kl(1, 1, 0, 2), tol=1e-4)
check('KL(N(0,4)||N(1,1))', kl_reverse, _kl(0, 2, 1, 1), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Careful: $\mathcal N(0, 4)$ has $\sigma = 2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Forward: $\ln 2 + \frac{1 + 1}{8} - \frac12 = 0.6931 + 0.25 - 0.5 = \mathbf{0.4431}$.
Reverse: $\ln\frac12 + \frac{4 + 1}{2} - \frac12 = -0.6931 + 2.5 - 0.5 = \mathbf{1.3069}$.
KL is not symmetric: approximating a wide distribution by a narrow one is penalised differently from the reverse. That choice (VI uses $q\Vert p$) shapes the approximation (Exercise 3).

```python
kl_forward = np.log(2) + 2 / 8 - 0.5
kl_reverse = np.log(0.5) + 5 / 2 - 0.5
```

</details>

### Exercise 8 · Bayesian linear regression with two data points
*✍️ By hand · ★★☆*

Model without bias: $w\sim\mathcal N(0, 1)$, $y_i\mid w\sim\mathcal N(wx_i, 1)$. Data $x = (1, 2)$, $y = (1, 3)$.
Using the conjugate update $\sigma_N^{-2} = \sigma_0^{-2} + \frac{1}{\sigma^2}\sum x_i^2$, $\mu_N = \sigma_N^2\frac{1}{\sigma^2}\sum x_iy_i$ (prior mean 0), compute the posterior
$\mathcal N(\mu_N, \sigma_N^2)$ and the posterior predictive mean and variance at $x_* = 3$. Which part of the predictive variance is aleatoric?

In [ ]:
mu_N = None
var_N = None
pred_mean = None
pred_var = None

# reference: brute-force posterior on a grid
_w = np.linspace(-6, 6, 200001); _lp = norm.logpdf(_w, 0, 1) + norm.logpdf(1, _w * 1, 1) + norm.logpdf(3, _w * 2, 1)
_p = np.exp(_lp - _lp.max()); _p /= _p.sum()
_m = (_w * _p).sum(); _v = ((_w - _m) ** 2 * _p).sum()
check('posterior mean', mu_N, _m, tol=1e-4); check('posterior variance', var_N, _v, tol=1e-4)
check('predictive mean at 3', pred_mean, 3 * _m, tol=1e-3); check('predictive variance at 3', pred_var, 1 + 9 * _v, tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\sum x_i^2 = 5$, $\sum x_iy_i = 7$. Predictive: $\mathcal N(\mu_Nx_*, \sigma^2 + x_*^2\sigma_N^2)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Precision $1 + 5 = 6$, so $\sigma_N^2 = \mathbf{1/6}$ and $\mu_N = 7/6\approx\mathbf{1.1667}$ (the least-squares slope $7/5 = 1.4$ shrunk towards the prior mean 0).
Predictive at $x_* = 3$: mean $3\cdot7/6 = \mathbf{3.5}$, variance $1 + 9/6 = \mathbf{2.5}$: $\sigma^2 = 1$ is aleatoric, $x_*^2\sigma_N^2 = 1.5$ epistemic (it grows with $|x_*|$).

```python
mu_N = 7 / 6
var_N = 1 / 6
pred_mean = 3.5
pred_var = 2.5
```

</details>

### Exercise 9 · MAP = regularised logistic regression
*✍️ By hand · ★★☆*

For Bayesian logistic regression with prior $w\sim\mathcal N(0, \sigma_0^2I)$ (bias included in $w$), write $\log p(w\mid\mathcal D)$ up to a constant and show that the MAP estimate minimises
cross-entropy $+\ \frac{1}{2\sigma_0^2}\lVert w\rVert^2$. Which value of scikit-learn's `C` corresponds to $\sigma_0^2$? Why is there still no closed-form posterior?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

scikit-learn minimises $\frac12\lVert w\rVert^2 + C\sum_n\ell_n(w)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\log p(w\mid\mathcal D) = \sum_n[y_n\log\pi_n + (1-y_n)\log(1-\pi_n)] - \frac{1}{2\sigma_0^2}\lVert w\rVert^2 + \text{const}$, $\pi_n = \sigma(w^Tx_n)$.
Maximising equals minimising $\text{CE}(w) + \frac{1}{2\sigma_0^2}\lVert w\rVert^2$; multiplying by $\sigma_0^2$ gives sklearn's form with $C = \sigma_0^2$. (This is the regulariser of
[[Lecture Notes 1-12#Lecture 2: Linear Models for Classification II]].)

The Bernoulli likelihood is not conjugate to the Gaussian prior, so the normaliser $\int p(\mathcal D\mid w)p(w)\,dw$ has no closed form: we need VI (Exercise 18), Laplace or MCMC.

</details>

### Exercise 10 · How many variational parameters?
*✍️ By hand · ★☆☆*

Count the latent variables in the BNN `model` of the note (`NHIDDEN = 250`, 1-D input and output, plus `log_sigma`). How many variational parameters does `AutoNormal` (one mean and one scale per latent)
create? How many would a full-rank Gaussian guide need (mean vector + lower-triangular Cholesky factor)?

In [ ]:
n_latent = None
n_autonormal = None
n_fullrank = None

_n = 1*250 + 250 + 250*1 + 1 + 1
check('latents', n_latent, _n)
check('AutoNormal', n_autonormal, 2 * _n)
check('full rank', n_fullrank, _n + _n * (_n + 1) // 2)

<details>
<summary><b>💡 Hint</b></summary>

Shapes: w1 (1,250), b1 (1,250), w2 (250,1), b2 (1,1), log_sigma scalar.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$250 + 250 + 250 + 1 + 1 = \mathbf{752}$ latents; `AutoNormal`: $2\cdot752 = \mathbf{1504}$; full rank: $752 + \frac{752\cdot753}{2} = 752 + 283\,128 = \mathbf{283\,880}$, the
$O(D^2)$ cost from the trade-off table.

```python
n_latent = 752
n_autonormal = 1504
n_fullrank = 752 + 752 * 753 // 2
```

</details>

### Exercise 11 · Mean-field VI shrinks the variance
*✍️ By hand · ★★★*

Let the true posterior be Gaussian, $p(w) = \mathcal N(0, \Sigma)$ with $\Sigma = \begin{pmatrix}1 & 0.9\\0.9 & 1\end{pmatrix}$ and precision $\Lambda = \Sigma^{-1}$.
For a mean-field Gaussian $q(w) = \mathcal N(w_1\mid m_1, s_1^2)\,\mathcal N(w_2\mid m_2, s_2^2)$, minimising $D_{\text{KL}}(q\Vert p)$ gives $m = 0$ and $s_i^2 = 1/\Lambda_{ii}$.
(a) Compute $\Lambda$ and the optimal $s_1^2$. (b) Compare with the true marginal variance $\Sigma_{11}$. (c) Sketch both: why is $q$ "too confident"?

In [ ]:
s1_sq = None

check('mean-field variance', s1_sq, 1 / np.linalg.inv(np.array([[1, .9], [.9, 1]]))[0, 0])

<details>
<summary><b>💡 Hint</b></summary>

$\Lambda = \frac{1}{1-0.81}\begin{pmatrix}1&-0.9\\-0.9&1\end{pmatrix}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\Lambda_{11} = 1/0.19\approx5.263$, so $s_1^2 = \mathbf{0.19}$. (b) The true marginal variance is $1$: mean-field reports a standard deviation of $0.436$ instead of $1$.
(c) $p$ is a long thin ellipse along $w_1 = w_2$; an axis-aligned $q$ must fit inside it (zero-forcing KL), so its width equals the ellipse's **conditional** width $1/\sqrt{\Lambda_{ii}}$,
not the marginal spread. The stronger the correlation, the worse the underestimate.

```python
s1_sq = 0.19
```

</details>

## Part C · Code from scratch

### Exercise 12 · KL divergence: closed form vs. Monte Carlo
*💻 Code · ★☆☆*

Implement `kl_gauss(m1, s1, m2, s2)` (closed form, $s$ = standard deviations) and `kl_mc(m1, s1, m2, s2, n)` that estimates $E_{q}[\log q - \log p]$ with $n$ samples from $q$.
Compare both on $q = \mathcal N(1, 1)$, $p = \mathcal N(0, 2^2)$ with $n = 100\,000$.

In [ ]:
def kl_gauss(m1, s1, m2, s2):
    return None

def kl_mc(m1, s1, m2, s2, n=100_000):
    return None

kl_cf = kl_gauss(1, 1, 0, 2)
kl_est = kl_mc(1, 1, 0, 2)

_ref = quad(lambda x: norm.pdf(x, 1, 1) * (norm.logpdf(x, 1, 1) - norm.logpdf(x, 0, 2)), -30, 30)[0]
check('closed form', kl_cf, _ref)
check('Monte Carlo (within 0.01)', kl_est, _ref, tol=1e-2)

<details>
<summary><b>💡 Hint</b></summary>

`x = rng.normal(m1, s1, n)`; `np.mean(norm.logpdf(x, m1, s1) - norm.logpdf(x, m2, s2))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The Monte Carlo estimator is unbiased with error $\propto1/\sqrt n$; this is exactly how SVI estimates ELBO terms that have no closed form (`Trace_ELBO` uses a few samples per step).

```python
def kl_gauss(m1, s1, m2, s2):
    return np.log(s2 / s1) + (s1 ** 2 + (m1 - m2) ** 2) / (2 * s2 ** 2) - 0.5

def kl_mc(m1, s1, m2, s2, n=100_000):
    x = rng.normal(m1, s1, n)
    return np.mean(norm.logpdf(x, m1, s1) - norm.logpdf(x, m2, s2))

kl_cf = kl_gauss(1, 1, 0, 2)
kl_est = kl_mc(1, 1, 0, 2)
print(kl_cf, kl_est)
```

</details>

### Exercise 13 · A plate is a vectorised sum
*💻 Code · ★☆☆*

For $x_i\sim\mathcal N(\mu,\sigma)$ i.i.d., compute the log-likelihood of 100 000 points (a) with a Python loop over data points (like the "sequential" Pyro version) and
(b) vectorised (like `pyro.plate`). Store both results and the speed-up factor (use `time.perf_counter`).

In [ ]:
import time
x_plate = rng.normal(2.0, 1.5, 100_000)
mu_p, sigma_p = 2.0, 1.5
ll_loop = None
ll_vec = None
speedup = None

check('loop = vectorised = scipy', [ll_loop, ll_vec] if ll_loop is not None else None, [norm.logpdf(x_plate, mu_p, sigma_p).sum()] * 2)

<details>
<summary><b>💡 Hint</b></summary>

$\log\mathcal N(x\mid\mu,\sigma) = -\frac12\log(2\pi\sigma^2) - \frac{(x-\mu)^2}{2\sigma^2}$; for the loop use `math.log` on each element.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Both give the same number; the vectorised version is typically 50 to 100 times faster. Pyro's `plate` additionally tells the inference engine that the terms are conditionally
independent, which is what allows mini-batch subsampling with the correct scaling.

```python
import time, math
x_plate = rng.normal(2.0, 1.5, 100_000)
mu_p, sigma_p = 2.0, 1.5
t0 = time.perf_counter()
ll_loop = 0.0
c = -0.5 * math.log(2 * math.pi * sigma_p ** 2)
for xi in x_plate:
    ll_loop += c - (xi - mu_p) ** 2 / (2 * sigma_p ** 2)
t1 = time.perf_counter()
ll_vec = np.sum(-0.5 * np.log(2 * np.pi * sigma_p ** 2) - (x_plate - mu_p) ** 2 / (2 * sigma_p ** 2))
t2 = time.perf_counter()
speedup = (t1 - t0) / (t2 - t1)
print('speed-up: %.0fx' % speedup)
```

</details>

### Exercise 14 · The Bayesian linear regression posterior
*💻 Code · ★★☆*

Implement `blr_posterior(X, y, mu0, Sigma0, sigma2)` returning $(\mu_N, \Sigma_N)$:
$$\Sigma_N = \left(\Sigma_0^{-1} + \tfrac{1}{\sigma^2}X^TX\right)^{-1},\qquad \mu_N = \Sigma_N\left(\Sigma_0^{-1}\mu_0 + \tfrac1{\sigma^2}X^Ty\right),$$
and `blr_predict(Xs, muN, SigmaN, sigma2)` returning predictive means and variances. Test on the data below with prior $\mathcal N(0, \tau^2I)$, $\tau^2 = 0.5$, $\sigma^2 = 0.25$:
the posterior mean must equal ridge regression with $\alpha = \sigma^2/\tau^2$.

In [ ]:
X_blr = np.c_[np.ones(30), rng.uniform(-2, 2, 30)]
y_blr = X_blr @ np.array([0.5, -1.0]) + 0.5 * rng.normal(size=30)
tau2, sigma2 = 0.5, 0.25

def blr_posterior(X, y, mu0, Sigma0, sigma2):
    return None, None

def blr_predict(Xs, muN, SigmaN, sigma2):
    return None, None

muN, SigmaN = blr_posterior(X_blr, y_blr, np.zeros(2), tau2 * np.eye(2), sigma2)

_r = Ridge(alpha=sigma2 / tau2, fit_intercept=False).fit(X_blr, y_blr)
check('posterior mean = ridge solution', muN, _r.coef_)
# posterior covariance = inverse Hessian of the negative log posterior (numerical Hessian)
_nlp = lambda w: np.sum((y_blr - X_blr @ w) ** 2) / (2 * sigma2) + w @ w / (2 * tau2)
_h = 1e-4; _H = np.zeros((2, 2)); _e = np.eye(2)
for i in range(2):
    for j in range(2):
        _H[i, j] = (_nlp(_r.coef_ + _h*_e[i] + _h*_e[j]) - _nlp(_r.coef_ + _h*_e[i] - _h*_e[j]) - _nlp(_r.coef_ - _h*_e[i] + _h*_e[j]) + _nlp(_r.coef_ - _h*_e[i] - _h*_e[j])) / (4 * _h * _h)
check('posterior covariance', SigmaN, np.linalg.inv(_H), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Use `np.linalg.inv` on the 2×2 precision (or `solve`). Predictive variance: `sigma2 + np.sum(Xs @ SigmaN * Xs, axis=1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With a zero-mean isotropic prior the posterior mean solves $(X^TX + \frac{\sigma^2}{\tau^2}I)w = X^Ty$: ridge regression. Bayesian learning adds the covariance $\Sigma_N$ on top, which
gives error bars (here the prior also shrinks the intercept, hence `fit_intercept=False` with a column of ones).

```python
X_blr = np.c_[np.ones(30), rng.uniform(-2, 2, 30)]
y_blr = X_blr @ np.array([0.5, -1.0]) + 0.5 * rng.normal(size=30)
tau2, sigma2 = 0.5, 0.25

def blr_posterior(X, y, mu0, Sigma0, sigma2):
    P0 = np.linalg.inv(Sigma0)
    SigmaN = np.linalg.inv(P0 + X.T @ X / sigma2)
    muN = SigmaN @ (P0 @ mu0 + X.T @ y / sigma2)
    return muN, SigmaN

def blr_predict(Xs, muN, SigmaN, sigma2):
    return Xs @ muN, sigma2 + np.sum(Xs @ SigmaN * Xs, axis=1)

muN, SigmaN = blr_posterior(X_blr, y_blr, np.zeros(2), tau2 * np.eye(2), sigma2)
print(muN.round(4), SigmaN.round(5))
```

</details>

### Exercise 15 · Epistemic uncertainty shrinks, aleatoric does not
*💻 Code · ★★☆*

Using `blr_posterior`, fit the model $y = 0.5 - x + \varepsilon$, $\varepsilon\sim\mathcal N(0, 0.5^2)$ on $N\in\{25, 100, 400, 10\,000\}$ points (prior $\mathcal N(0, I)$, known $\sigma^2 = 0.25$).
Store the posterior standard deviation of the slope in `slope_sd` (dict by $N$) and the predictive variance at $x = 1$ in `pred_var_at_1`. What do they converge to?

In [ ]:
Ns = [25, 100, 400, 10_000]
slope_sd = None
pred_var_at_1 = None

if slope_sd is not None:
    check('4x data halves the epistemic sd (ratio ~2)', abs(slope_sd[25] / slope_sd[100] - 2) < 0.5, True)
    check('predictive variance -> sigma^2 = 0.25', pred_var_at_1[10_000], 0.25, tol=2e-3)
else:
    check('4x data halves the epistemic sd (ratio ~2)', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Draw fresh data for each $N$ with a seeded generator; `pred_var = sigma2 + x^T SigmaN x` with `x = [1, 1]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The slope's posterior sd falls like $1/\sqrt N$ (epistemic, reducible), while the predictive variance converges to the noise variance $\sigma^2 = 0.25$ (aleatoric floor): more data cannot
make a noisy sensor less noisy.

```python
Ns = [25, 100, 400, 10_000]
slope_sd, pred_var_at_1 = {}, {}
r = np.random.default_rng(1)
for n in Ns:
    x = r.uniform(-2, 2, n); X = np.c_[np.ones(n), x]
    y = 0.5 - x + 0.5 * r.normal(size=n)
    m, S = blr_posterior(X, y, np.zeros(2), np.eye(2), 0.25)
    slope_sd[n] = np.sqrt(S[1, 1])
    pred_var_at_1[n] = blr_predict(np.array([[1., 1.]]), m, S, 0.25)[1][0]
print({n: round(v, 4) for n, v in slope_sd.items()}); print({n: round(v, 4) for n, v in pred_var_at_1.items()})
```

</details>

### Exercise 16 · Uncertainty in a data gap
*💻 Code · ★★☆*

Reproduce the "observation gap" experiment with a Bayesian **basis-function** regression (a cheap stand-in for a BNN): inputs in $[-2, 0]\cup[5, 7]$, none in between,
$y = \sin(x) + \mathcal N(0, 0.1^2)$. Features: 15 Gaussian bumps $\exp(-(x-c_j)^2/2)$ with centres $c_j$ evenly spaced in $[-3, 8]$ plus a constant; prior $\mathcal N(0, I)$, $\sigma^2 = 0.01$.
Plot the predictive mean ± 2 sd and a few posterior function samples on $[-3, 8]$. Store the predictive sd at $x = 2.5$ (`sd_gap`) and the mean predictive sd over the training inputs (`sd_train`).

In [ ]:
r = np.random.default_rng(2)
x_gap = np.r_[r.uniform(-2, 0, 20), r.uniform(5, 7, 20)]
y_gap = np.sin(x_gap) + 0.1 * r.normal(size=40)
centres = np.linspace(-3, 8, 15)

def features(x):
    return None   # (N, 16)

sd_gap = None
sd_train = None

if sd_gap is not None:
    check('gap is far more uncertain (> 3x)', sd_gap > 3 * sd_train, True)
else:
    check('gap is far more uncertain (> 3x)', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`np.c_[np.ones(len(x)), np.exp(-(x[:, None] - centres[None]) ** 2 / 2)]`. Function samples: draw $w\sim\mathcal N(\mu_N,\Sigma_N)$ with `rng.multivariate_normal`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Inside the data the sampled functions agree and the sd is close to the noise level $0.1$; in the gap they fan out and the predictive sd is several times larger (epistemic
uncertainty), exactly the warning the BNN in the note gives. A point-estimate (MLE/MAP) model would just draw one confident curve through the gap.

```python
r = np.random.default_rng(2)
x_gap = np.r_[r.uniform(-2, 0, 20), r.uniform(5, 7, 20)]
y_gap = np.sin(x_gap) + 0.1 * r.normal(size=40)
centres = np.linspace(-3, 8, 15)

def features(x):
    return np.c_[np.ones(len(x)), np.exp(-(x[:, None] - centres[None]) ** 2 / 2)]

mG, SG = blr_posterior(features(x_gap), y_gap, np.zeros(16), np.eye(16), 0.01)
xs = np.linspace(-3, 8, 300)
mean_s, var_s = blr_predict(features(xs), mG, SG, 0.01)
sd_gap = np.sqrt(blr_predict(features(np.array([2.5])), mG, SG, 0.01)[1][0])
sd_train = np.sqrt(blr_predict(features(x_gap), mG, SG, 0.01)[1]).mean()
print('sd in gap %.3f, sd on training inputs %.3f' % (sd_gap, sd_train))
plt.figure(figsize=(6, 3))
plt.fill_between(xs, mean_s - 2 * np.sqrt(var_s), mean_s + 2 * np.sqrt(var_s), alpha=0.25)
for w in r.multivariate_normal(mG, SG, size=5):
    plt.plot(xs, features(xs) @ w, lw=0.8)
plt.plot(x_gap, y_gap, 'k.'); plt.show()
```

</details>

### Exercise 17 · The ELBO of a conjugate model: analytic and Monte Carlo
*💻 Code · ★★★*

Model: $w\sim\mathcal N(0,\tau^2)$, $y_i\mid w\sim\mathcal N(wx_i,\sigma^2)$; guide $q(w) = \mathcal N(m, s^2)$. With $\tau^2 = 1$, $\sigma^2 = 1$ and the data of Exercise 8:

1. Implement `elbo_analytic(m, s)` using $E_q[(y_i - wx_i)^2] = (y_i - mx_i)^2 + x_i^2s^2$, $E_q[w^2] = m^2+s^2$ and the Gaussian entropy $\frac12\log(2\pi e s^2)$.
2. Implement `elbo_mc(m, s, n)` by sampling $w\sim q$ and averaging $\log p(\mathcal D, w) - \log q(w)$.
3. Compute the exact log evidence ($\mathbf{y}\sim\mathcal N(0, \sigma^2I + \tau^2\mathbf{x}\mathbf{x}^T)$) in `log_evidence`.
4. Verify: ELBO at the exact posterior ($m = 7/6$, $s^2 = 1/6$) equals the log evidence, and is smaller for $q = \mathcal N(0, 1)$.

In [ ]:
x8, y8 = np.array([1., 2.]), np.array([1., 3.])

def elbo_analytic(m, s):
    return None

def elbo_mc(m, s, n=200_000):
    return None

log_evidence = None

_le = multivariate_normal(np.zeros(2), np.eye(2) + np.outer(x8, x8)).logpdf(y8)
check('log evidence', log_evidence, _le)
if elbo_analytic(0, 1) is not None:
    check('ELBO(exact posterior) = log evidence', elbo_analytic(7 / 6, np.sqrt(1 / 6)), _le)
    check('ELBO(prior as q) < log evidence', elbo_analytic(0, 1) < _le, True)
    check('Monte Carlo ELBO ~ analytic', elbo_mc(0.5, 0.7), elbo_analytic(0.5, 0.7), tol=2e-2)
else:
    check('ELBO(exact posterior) = log evidence', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

$\text{ELBO} = \sum_iE_q[\log\mathcal N(y_i\mid wx_i,\sigma^2)] + E_q[\log\mathcal N(w\mid0,\tau^2)] + \mathbb H[q]$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

At the exact posterior the KL term is zero, so ELBO = $\log p(\mathcal D)\approx-3.650$ to machine precision; with $q$ equal to the prior the ELBO is lower by exactly
$D_{\text{KL}}(q\Vert p(w\mid\mathcal D))$. The Monte Carlo estimate agrees within its sampling error: what SVI optimises with noisy gradients.

```python
x8, y8 = np.array([1., 2.]), np.array([1., 3.])

def elbo_analytic(m, s):
    lik = np.sum(-0.5 * np.log(2 * np.pi) - ((y8 - m * x8) ** 2 + x8 ** 2 * s ** 2) / 2)
    prior = -0.5 * np.log(2 * np.pi) - (m ** 2 + s ** 2) / 2
    entropy = 0.5 * np.log(2 * np.pi * np.e * s ** 2)
    return lik + prior + entropy

def elbo_mc(m, s, n=200_000):
    w = rng.normal(m, s, n)
    logjoint = norm.logpdf(w, 0, 1) + norm.logpdf(y8[None], w[:, None] * x8[None], 1).sum(1)
    return np.mean(logjoint - norm.logpdf(w, m, s))

log_evidence = multivariate_normal(np.zeros(2), np.eye(2) + np.outer(x8, x8)).logpdf(y8)
print('log evidence %.4f, ELBO(posterior) %.4f, ELBO(prior) %.4f' % (log_evidence, elbo_analytic(7/6, np.sqrt(1/6)), elbo_analytic(0, 1)))
```

</details>

### Exercise 18 · Stochastic variational inference for Bayesian logistic regression
*💻 Code · ★★★*

Implement a mean-field Gaussian guide $q(w) = \prod_k\mathcal N(\mu_k, e^{2\rho_k})$ (like `AutoNormal`) for Bayesian logistic regression with prior $\mathcal N(0, I)$ on the data below
(bias as a column of ones). Use the **reparameterisation trick** $w = \mu + e^{\rho}\odot\varepsilon$, $\varepsilon\sim\mathcal N(0, I)$:
$$\nabla_\mu\text{ELBO}\approx\overline{g},\qquad \nabla_\rho\text{ELBO}\approx\overline{g\odot\varepsilon}\odot e^{\rho} + 1,\qquad g = \nabla_w\log p(\mathcal D, w) = X^T(y-\sigma(Xw)) - w,$$
(the $+1$ is the gradient of the entropy $\sum_k\rho_k$). Run 4000 steps of gradient **ascent** (lr 0.002, 10 samples per step, start $\mu = 0$, $\rho = -1$) and average the
parameters over the last 1000 steps. Store `mu_vi` and `sd_vi`, and compare with the MAP (`LogisticRegression(C=1, fit_intercept=False)`) and with the Laplace approximation
$\text{sd} = \sqrt{\operatorname{diag}(H^{-1})}$, $H = X^T\operatorname{diag}(\pi(1-\pi))X + I$ at the MAP.

In [ ]:
r = np.random.default_rng(6)
X_bl = np.c_[np.ones(200), r.normal(size=(200, 2))]
y_bl = (r.random(200) < expit(X_bl @ np.array([-0.5, 2.0, -1.0]))).astype(float)
mu_vi = None
sd_vi = None

_m = LogisticRegression(C=1.0, fit_intercept=False, tol=1e-10).fit(X_bl, y_bl).coef_[0]
_p = expit(X_bl @ _m); _H = (X_bl * (_p * (1 - _p))[:, None]).T @ X_bl + np.eye(3)
if mu_vi is not None:
    check('VI mean close to the MAP (within 0.1)', np.max(np.abs(mu_vi - _m)) < 0.1, True)
    check('mean-field sd ~ 1/sqrt(H_kk) (within 15%)', sd_vi, 1 / np.sqrt(np.diag(_H)), tol=0.15 * 0.2)
    check('mean-field sd < Laplace (full-covariance) sd', bool(np.all(sd_vi < np.sqrt(np.diag(np.linalg.inv(_H))) + 0.01)), True)
else:
    check('VI mean close to the MAP (within 0.1)', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Vectorise over samples: `W = mu + np.exp(rho) * eps` with `eps` of shape (10, 3); then `G = (y - expit(W @ X.T)) @ X - W` has shape (10, 3).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Use a separate seeded generator for `eps` so the run is reproducible.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The VI means land close to the MAP (with 200 points the posterior is nearly Gaussian and symmetric). The mean-field standard deviations match $1/\sqrt{H_{kk}}$, the **conditional**
widths, and are smaller than the Laplace marginal sds $\sqrt{(H^{-1})_{kk}}$, the variance underestimation of Exercise 11 in a real model.
This loop is what `SVI(model, AutoNormal(model), optim, Trace_ELBO()).step()` does, with autograd computing $g$ and Adam choosing the steps.

```python
r = np.random.default_rng(6)
X_bl = np.c_[np.ones(200), r.normal(size=(200, 2))]
y_bl = (r.random(200) < expit(X_bl @ np.array([-0.5, 2.0, -1.0]))).astype(float)

mu, rho = np.zeros(3), np.full(3, -1.0)
r_eps = np.random.default_rng(0)
hist = []
for t in range(4000):
    eps = r_eps.normal(size=(10, 3))
    W = mu + np.exp(rho) * eps
    G = (y_bl - expit(W @ X_bl.T)) @ X_bl - W          # grad of log joint, one row per sample
    mu = mu + 0.002 * G.mean(0)
    rho = rho + 0.002 * ((G * eps).mean(0) * np.exp(rho) + 1)
    if t >= 3000:
        hist.append(np.r_[mu, rho])
h = np.mean(hist, 0)
mu_vi, sd_vi = h[:3], np.exp(h[3:])
print('VI mean', mu_vi.round(3), ' VI sd', sd_vi.round(3))
```

</details>

### Exercise 19 · A posterior predictive check
*💻 Code · ★★☆*

Model: $x_i\sim\mathcal N(\mu, \sigma^2)$ with $\sigma$ fixed to the sample sd and a flat prior on $\mu$ (posterior $\mu\mid\mathcal D\sim\mathcal N(\bar x, \sigma^2/N)$).
Implement `ppc_pvalue(x, stat, n_rep)`: draw $\mu$ from the posterior, simulate a replicated dataset of size $N$, compute `stat`, and return the fraction of replicas with
$T(\mathcal X^{\text{rep}})\ge T(\mathcal X)$. Apply it with $T = $ skewness to (a) Gaussian data and (b) exponential data (both $N = 200$, given below).

In [ ]:
r = np.random.default_rng(3)
x_normal = r.normal(1.0, 1.0, 200)
x_expo = r.exponential(1.0, 200)

def ppc_pvalue(x, stat, n_rep=2000, seed=0):
    return None

p_normal = ppc_pvalue(x_normal, skew)
p_expo = ppc_pvalue(x_expo, skew)

if p_normal is not None:
    check('Gaussian data: no evidence of misfit (p > 0.05)', 0.05 < p_normal < 0.95, True)
    check('exponential data: misfit detected (p < 0.01)', p_expo < 0.01, True)
else:
    check('Gaussian data: no evidence of misfit (p > 0.05)', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Vectorise: `mus = r.normal(xbar, s/np.sqrt(N), n_rep)`; `rep = r.normal(mus[:, None], s, (n_rep, N))`; `skew(rep, axis=1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

For the Gaussian data the observed skewness is typical of the replicas (a moderate p-value); for the exponential data (skewness about 2) no symmetric replica comes close and $p\approx0$.
In Box's loop this criticism says: revise the likelihood (e.g. a log-normal or gamma model) and go round again.

```python
r = np.random.default_rng(3)
x_normal = r.normal(1.0, 1.0, 200)
x_expo = r.exponential(1.0, 200)

def ppc_pvalue(x, stat, n_rep=2000, seed=0):
    g = np.random.default_rng(seed)
    N, s = len(x), x.std()
    mus = g.normal(x.mean(), s / np.sqrt(N), n_rep)
    rep = g.normal(mus[:, None], s, (n_rep, N))
    return np.mean(stat(rep, axis=1) >= stat(x))

p_normal = ppc_pvalue(x_normal, skew)
p_expo = ppc_pvalue(x_expo, skew)
print('p (Gaussian data) = %.3f, p (exponential data) = %.4f' % (p_normal, p_expo))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).*